# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [1]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

HypEx 1.0.7 · pandas 2.2.3


### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [2]:
VARIANT = None   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [3]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

вариант 53: «Бонус 300 ₽ за первую покупку в доставке»
6000 клиентов · test 3017 · control 2983


### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [4]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

проверка формата готова


## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [5]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

клиентов на группу: 3000, σ оборота: 3198 ₽, ρ двух прошлых периодов: 0.592
MDE без снижения дисперсии: 231 ₽
MDE с CUPED:                187 ₽
+3 % к среднему обороту:    96 ₽


#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Акция «Бонус 300 ₽ за первую покупку в доставке» подтолкнёт клиента к первой транзакции и заодно разгонит остальные покупки. Ждём, что средний оборот в партнёрских сервисах на аккаунт подписки за 4 недели вырастет на 3% - при среднем обороте предпериода около 3 200 ₽ это примерно +96 ₽ на клиента |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Аккаунт подписки (в выгрузке это одна строка и один client_id). Подпиской пользуются несколько человек, это видно по колонке members, поэтому рандомизировать людей внутри подписки нельзя: члены одной семьи попали бы в разные группы, контрольная группа увидела бы чужую акцию, и эффект перетёк бы через границу группы. Рандомизируем аккаунт целиком, тогда в каждой группе оказываются целиком независимые домохозяйства |
| группы и доли | 50/50: тест получает предложение, контроль живёт как обычно. На 6 000 аккаунтов это по 3 000 на группу |
| основная метрика | spend — оборот в партнёрских сервисах на аккаунт за 4 недели эксперимента, ₽ |
| метрики-стражи: что не должно ухудшиться | churned (ушёл из подписки за время эксперимента) и tickets (число обращений в поддержку на клиента). Отток не должен вырасти более чем на 0,5 п.п., обращения — более чем на 0,02 на клиента. Про маржинальность и конверсию в расширенный пакет в этих данных судить нельзя: таких колонок в выгрузке нет, так что в решающее правило их не включаем |
| α, мощность, MDE - числами из ячейки выше | α = 0,05, мощность 80%, одно сравнение по основной метрике. По mde_for_n при σ = 3 198 ₽ и 3 000 клиентов на группу эксперимент замечает 231 ₽ без снижения дисперсии и 187 ₽ с CUPED. Плановые 96 ₽ в этот зазор не попадают: по n_per_group под них нужно около 22 600 клиентов на группу с CUPED (34 800 без) |
| поправка на множественные сравнения | Не требуется для основной проверки, поскольку используется одна основная метрика. |
| оценщик | Разность средних spend (test − control), сниженная по CUPED на ковариате spend_prev; θ оценивается по всей выборке. Интервал через нормальное приближение, сверка с HypEx и с CUPED внутри HypEx. Бутстреп и мэтчинг считаем как проверки, а не как основные оценки |
| окно | 4 недели с старта акции. Смотрим один раз по итогам окна, промежуточные снятия не делаем |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | Запускаем, если нижняя граница 95% интервала после CUPED выше 0 ₽ и при этом нижняя граница в пересчёте на деньги тоже выше 0 ₽, а стражи не выбились за пороги и после Холма ни один не выглядит ухудшившимся. Отказываемся, если разность отрицательная, или весь интервал по деньгам ушёл ниже нуля, или отток вырос на 0,5 п.п. и больше при значимом результате, или обращения в поддержку перевешивают комиссию с прироста. Дорабатываем и повторяем с бóльшим объёмом, если разность положительная, но ноль внутри интервала: это не провал и не повод запускать, это значит, что мощности на плановые 96 ₽ не хватило |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [6]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes.tolist())

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

{'control': 2983, 'test': 3017}
scipy:    p = 0.6607
pilot.py: p = 0.6607


0.661

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [7]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

spend_prev: test 3179.15 ₽, control 3200.35 ₽
p = 0.7975


0.797

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [8]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

test 3245.84 ₽, control 3213.53 ₽
разность 32.31 ₽, интервал -132.12 … 196.74, p = 0.7001


[32.31, -132.12, 196.74]

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [9]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

feature group  control mean   test mean  difference  difference % TTest pass  TTest p-value UTest pass  UTest p-value
  spend  test   3213.529933 3245.840272   32.310339      1.005447     NOT OK       0.700148     NOT OK       0.582365


[0.7001, 0.5824]

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [10]:
# Задача 1: проверка распределения по группам (SRM)
n_control = len(control)
n_test = len(test)
p_srm = srm_pvalue([n_control, n_test])

# Задача 2: проверка баланса по обороту за предпериод
balance = diff_ci(test["spend_prev"], control["spend_prev"])

# Задача 3: эффект акции по основному показателю
effect = diff_ci(test["spend"], control["spend"])

# Задача 4: результаты t-test и U-test
p_ttest = float(ab["TTest p-value"].iloc[0])
p_utest = float(ab["UTest p-value"].iloc[0])

# Дизайн эксперимента
alpha = 0.05

print("ВОПРОС 1. ПРОВЕРКА ЭКСПЕРИМЕНТА")
print("-" * 50)

print("\n1. Распределение групп:")
print(f"Control: {n_control}, Test: {n_test}")
print(f"SRM p-value: {p_srm:.4f}")
print(f"Разбиение корректно: {p_srm >= alpha}")

print("\n2. Баланс по предпериодному обороту:")
print(f"Разница test − control: {balance['diff']:.2f} ₽")
print(f"p-value: {balance['p']:.4f}")
print(f"Статистически значимое различие: {balance['p'] < alpha}")

print("\n3. Эффект эксперимента:")
print(f"Разница test − control: {effect['diff']:.2f} ₽")
print(f"95% ДИ: [{effect['lo']:.2f}; {effect['hi']:.2f}] ₽")
print(f"p-value: {effect['p']:.4f}")
print(f"Интервал включает ноль: {effect['lo'] <= 0 <= effect['hi']}")
print(f"MDE без CUPED: {mde_raw:.2f} ₽")
print(f"MDE с CUPED: {mde_cuped:.2f} ₽")

print("\n4. Сравнение статистических критериев:")
print(f"t-test p-value: {p_ttest:.4f}")
print(f"U-test p-value: {p_utest:.4f}")
print(f"Выводы совпадают: {(p_ttest < alpha) == (p_utest < alpha)}")

ВОПРОС 1. ПРОВЕРКА ЭКСПЕРИМЕНТА
--------------------------------------------------

1. Распределение групп:
Control: 2983, Test: 3017
SRM p-value: 0.6607
Разбиение корректно: True

2. Баланс по предпериодному обороту:
Разница test − control: -21.20 ₽
p-value: 0.7974
Статистически значимое различие: False

3. Эффект эксперимента:
Разница test − control: 32.31 ₽
95% ДИ: [-132.12; 196.74] ₽
p-value: 0.7001
Интервал включает ноль: True
MDE без CUPED: 231.35 ₽
MDE с CUPED: 186.52 ₽

4. Сравнение статистических критериев:
t-test p-value: 0.7001
U-test p-value: 0.5824
Выводы совпадают: True


#### ✍️ Ответ на вопрос 1

Разбиению доверять можно: 3 017 против 2 983, SRM даёт p = 0,661, и по обороту за 4 недели до старта группы совпали (p = 0,797 при разнице 21,20 ₽ не в пользу теста). Прирост вышел 32,31 ₽ с 95% интервалом от −132,12 до 196,74 ₽; ноль внутри интервала, так что утверждать рост нельзя. Эффект и весь интервал заметно меньше MDE в 187 ₽ (без CUPED 231 ₽), то есть эксперимент и не был рассчитан заметить плановые 96 ₽ - при 3 000 на группу мощность на 96 ₽ получается около трети. t-критерий (p = 0,7001) и U-критерий (p = 0,5824) сошлись в выводе «не значимо», поэтому расхождения между ними в этом запуске и не было.

## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [11]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table)[1]

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

         control    test    diff      lo      hi       p
metric                                                  
churned   0.0285  0.0292  0.0007 -0.0078  0.0091  0.8762
tickets   0.0654  0.0583 -0.0070 -0.0196  0.0056  0.2742
churned     0   1
grp              
control  2898  85
test     2929  88
χ² по оттоку: p = 0.9373


[0.07, 0.937, -0.007, 0.274]

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [12]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

HypEx о поправке: There was less than three groups or multitest method wasn't provided
  feature  control mean  test mean  difference  TTest p-value  holm  rejected
    spend     3213.5299  3245.8403     32.3103         0.7001   1.0     False
purchases        4.8907     4.9450      0.0543         0.6194   1.0     False
  churned        0.0285     0.0292      0.0007         0.8762   1.0     False
  tickets        0.0654     0.0583     -0.0070         0.2742   1.0     False


[]

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [13]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

theta = 0.598, дисперсия ниже на 34.7%
разность 44.98 ₽, интервал -87.94 … 177.91


[0.598, 34.7, 44.98, -87.94, 177.91]

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [14]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

target covariate variance_reduction
 spend spend_lag              34.7%
    feature group  control mean   test mean  difference  difference % TTest pass  TTest p-value
      spend  test   3213.529933 3245.840272   32.310339      1.005447     NOT OK       0.700148
spend_cuped  test   3207.156794 3252.141589   44.984795      1.402638     NOT OK       0.507155


44.98

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [15]:
import inspect

print("Функция diff_ci загружена из:")
print(diff_ci.__module__)

print("\nТекущий код diff_ci:")
print(inspect.getsource(diff_ci))

print("\nПроверка исходных данных:")
for metric in ("churned", "tickets"):
    print(f"\n{metric}")
    print("Тип:", mine[metric].dtype)
    print("Control mean:", control[metric].mean())
    print("Test mean:", test[metric].mean())
    print("Control variance:", control[metric].var())
    print("Test variance:", test[metric].var())
    print("Результат diff_ci:", diff_ci(test[metric], control[metric]))

Функция diff_ci загружена из:
prime.pilot

Текущий код diff_ci:
def diff_ci(t: pd.Series, c: pd.Series, alpha: float = 0.05) -> dict:
    """Разность средних test − control, её 95 % интервал и p-значение."""
    d = t.mean() - c.mean()
    # Стандартная ошибка разности: дисперсия каждой группы делится на её размер.
    se = (t.var() / len(t) + c.var() / len(c)) ** 0.5
    q = NormalDist().inv_cdf(1 - alpha / 2)
    p = 2 * (1 - NormalDist().cdf(abs(d / se)))
    return {"diff": d, "lo": d - q * se, "hi": d + q * se, "p": p}


Проверка исходных данных:

churned
Тип: int64
Control mean: 0.02849480388870265
Test mean: 0.02916804772953265
Control variance: 0.027692133356626517
Test variance: 0.028326661737334592
Результат diff_ci: {'diff': 0.0006732438408299998, 'lo': -0.00779605349029041, 'hi': 0.00914254117195041, 'p': 0.8761890738120859}

tickets
Тип: int64
Control mean: 0.06537043245055313
Test mean: 0.0583360954590653
Control variance: 0.06648315414894104
Test variance: 0.057603729177

In [16]:
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
# Задача 5: метрики-стражи
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(),
     "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
])

table = pd.crosstab(mine["grp"], mine["churned"])
p_churn = chi2_contingency(table)[1]

# Задача 6: поправка Холма
p_values = metrics["TTest p-value"]
holm_p = multipletests(p_values, method="holm")[1]
significant_after_holm = metrics.loc[holm_p < alpha, "feature"].tolist()

# Задача 3: обычный эффект
effect_raw = diff_ci(test["spend"], control["spend"])

# Задача 7: эффект после CUPED
mine["spend_cuped"], theta = cuped(
    mine["spend"], mine["spend_prev"]
)

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]
effect_adjusted = diff_ci(
    test["spend_cuped"], control["spend_cuped"]
)

# Сравнение ширины доверительных интервалов
width_raw = effect_raw["hi"] - effect_raw["lo"]
width_cuped = effect_adjusted["hi"] - effect_adjusted["lo"]
ci_reduction = (1 - width_cuped / width_raw) * 100

# Предыдущий оборот
previous = diff_ci(test["spend_prev"], control["spend_prev"])

print("МЕТРИКИ-СТРАЖИ")
print(guards.round(4).to_string(index=False))
print(f"Отток: χ² p-value = {p_churn:.4f}")

print("\nПОПРАВКА ХОЛМА")
print("Значимые метрики:", significant_after_holm)

print("\nCUPED")
print(f"Дисперсия снижена: {reduction:.1%}")
print(f"Обычный эффект: {effect_raw['diff']:.2f} ₽")
print(f"Обычный 95% ДИ: [{effect_raw['lo']:.2f}; {effect_raw['hi']:.2f}]")
print(f"CUPED эффект: {effect_adjusted['diff']:.2f} ₽")
print(f"CUPED 95% ДИ: [{effect_adjusted['lo']:.2f}; {effect_adjusted['hi']:.2f}]")
print(f"Сужение интервала: {ci_reduction:.1f}%")

print("\nПРЕДЫДУЩИЙ ОБОРОТ")
print(f"Test: {test['spend_prev'].mean():.2f} ₽")
print(f"Control: {control['spend_prev'].mean():.2f} ₽")
print(f"p-value: {previous['p']:.4f}")

МЕТРИКИ-СТРАЖИ
 metric  control   test    diff      lo     hi      p
churned   0.0285 0.0292  0.0007 -0.0078 0.0091 0.8762
tickets   0.0654 0.0583 -0.0070 -0.0196 0.0056 0.2742
Отток: χ² p-value = 0.9373

ПОПРАВКА ХОЛМА
Значимые метрики: []

CUPED
Дисперсия снижена: 34.7%
Обычный эффект: 32.31 ₽
Обычный 95% ДИ: [-132.12; 196.74]
CUPED эффект: 44.98 ₽
CUPED 95% ДИ: [-87.94; 177.91]
Сужение интервала: 19.2%

ПРЕДЫДУЩИЙ ОБОРОТ
Test: 3179.15 ₽
Control: 3200.35 ₽
p-value: 0.7974


#### ✍️ Ответ на вопрос 2

Стражи чистые: отток +0,07 п.п. (2,92% против 2,85%, χ² p = 0,937), обращений стало меньше на 0,007 на клиента (p = 0,274) - оба далеки от порогов дизайн-документа. После поправки Холма по четырём метрикам не осталось ни одной значимой, список пустой. CUPED снял 34,7% дисперсии и сузил интервал на 19,2% - с 328,85 ₽ до 265,85 ₽, оценка при этом сдвинулась с 32,31 до 44,98 ₽, и сдвиг объясним ровно ковариатой: 32,31 + 0,598 × 21,20 = 44,98 ₽, то есть тестовая группа до старта тратила чуть меньше, и CUPED это различие вернул в оценку. Значимее от этого ничего не стало: p по CUPED 0,507, ноль так и остался в интервале от −87,94 до 177,91 ₽.

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [17]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.

rng = np.random.default_rng(VARIANT)

x = test["spend"].to_numpy()
y = control["spend"].to_numpy()

diffs = []

for _ in range(2000):
    test_sample = rng.choice(x, size=len(x), replace=True)
    control_sample = rng.choice(y, size=len(y), replace=True)

    diffs.append(test_sample.mean() - control_sample.mean())

bootstrap_ci = np.percentile(diffs, [2.5, 97.5])

print(f"95% бутстреп-интервал: [{bootstrap_ci[0]:.2f}; {bootstrap_ci[1]:.2f}] ₽")

answer_9 = [round(float(bootstrap_ci[0]), 2),
            round(float(bootstrap_ci[1]), 2)]

answer_9

95% бутстреп-интервал: [-136.22; 206.24] ₽


[-136.22, 206.24]

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [18]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.

candidates = {
    "spend_prev": mine["spend_prev"],
    "spend_prev2": mine["spend_prev2"],
    "spend_prev+spend_prev2": mine["spend_prev"] + mine["spend_prev2"],
    "purchases_prev": mine["purchases_prev"],
    "sessions_prev": mine["sessions_prev"],
    "sessions_prev2": mine["sessions_prev2"],
}

results = {}

for name, x in candidates.items():
    rho = mine["spend"].corr(x)
    results[name] = rho ** 2 * 100

# Отдельно проверяем purchases за время эксперимента
rho_purchases = mine["spend"].corr(mine["purchases"])
purchases_r2 = rho_purchases ** 2 * 100

results_df = pd.DataFrame.from_dict(
    results, orient="index", columns=["rho_squared_percent"]
).sort_values("rho_squared_percent", ascending=False)

print("Кандидаты из предпериода:")
print(results_df.round(1).to_string())

print(f"\npurchases (во время эксперимента): {purchases_r2:.1f}%")

best_name = results_df.index[0]
best_value = results_df.iloc[0, 0]

answer_10 = [best_name, round(float(best_value), 1)]
answer_10

Кандидаты из предпериода:
                        rho_squared_percent
spend_prev+spend_prev2                 41.5
sessions_prev                          41.2
sessions_prev2                         35.4
spend_prev                             34.7
spend_prev2                            31.5
purchases_prev                         29.0

purchases (во время эксперимента): 70.8%


['spend_prev+spend_prev2', 41.5]

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [19]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.

features = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]

match_data = mine[["client_id", "grp", "spend"] + features].copy()
match_data["t"] = (match_data["grp"] == "test").astype(int)
match_data = match_data.drop(columns="grp")

roles = {"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
         **{col: FeatureRole() for col in features}}
ds_match = Dataset(roles=roles, data=match_data, default_role=FeatureRole())

resume = F(Matching(n_neighbors=5).execute(ds_match).resume).reset_index()
ate = resume.loc[resume.astype(str).isin(["ATE"]).any(axis=1)].iloc[0]
print(ate.to_frame().T.to_string(index=False))

answer_11 = [round(float(ate["Effect Size"]), 2),
             round(float(ate["CI Lower"]), 2),
             round(float(ate["CI Upper"]), 2)]
answer_11

index Effect Size Standard Error P-value CI Lower CI Upper outcome
  ATE       36.77          69.62     0.6   -99.69   173.23   spend


[36.77, -99.69, 173.23]

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [20]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽

# Эффект оборота после CUPED
mine["spend_cuped"], theta = cuped(
    mine["spend"], mine["spend_prev"]
)

g = mine.groupby("grp")
effect_cuped = diff_ci(
    g.get_group("test")["spend_cuped"],
    g.get_group("control")["spend_cuped"]
)

# Разница по обращениям и оттоку
tickets_effect = diff_ci(
    g.get_group("test")["tickets"],
    g.get_group("control")["tickets"]
)["diff"]

churn_effect = diff_ci(
    g.get_group("test")["churned"],
    g.get_group("control")["churned"]
)["diff"]

# Денежные составляющие
commission_income = effect_cuped["diff"] * COMMISSION
tickets_cost = tickets_effect * TICKET_COST
churn_cost = churn_effect * CHURN_COST

total = commission_income - tickets_cost - churn_cost

print(f"Комиссия с прироста оборота: {commission_income:+.2f} ₽")
print(f"Экономия на обращениях ({tickets_effect:+.3f} × {TICKET_COST}): {-tickets_cost:+.2f} ₽")
print(f"Потери из-за оттока ({churn_effect:+.4f} × {CHURN_COST}): {-churn_cost:+.2f} ₽")
print("-" * 40)
print(f"ИТОГО: {total:+.2f} ₽ на клиента")

answer_12 = round(float(total), 2)
answer_12

Комиссия с прироста оборота: +0.67 ₽
Экономия на обращениях (-0.007 × 150): +1.06 ₽
Потери из-за оттока (+0.0007 × 750): -0.50 ₽
----------------------------------------
ИТОГО: +1.22 ₽ на клиента


1.22

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [21]:

import numpy as np
from hypex import Matching
from hypex.dataset import Dataset, InfoRole, TargetRole, TreatmentRole, FeatureRole

# 1. Обычная оценка эффекта (задача 3)
raw_effect = diff_ci(test["spend"], control["spend"])
raw_ci = (raw_effect["lo"], raw_effect["hi"])
raw_width = raw_ci[1] - raw_ci[0]

# 2. CUPED (задача 7)
spend_cuped, theta = cuped(mine["spend"], mine["spend_prev"])
mine["spend_cuped"] = spend_cuped

g = mine.groupby("grp")
cuped_effect = diff_ci(
    g.get_group("test")["spend_cuped"],
    g.get_group("control")["spend_cuped"]
)
cuped_ci = (cuped_effect["lo"], cuped_effect["hi"])
cuped_width = cuped_ci[1] - cuped_ci[0]

# 3. Бутстреп (задача 9)
rng = np.random.default_rng(VARIANT)
x = test["spend"].to_numpy()
y = control["spend"].to_numpy()

boot_diffs = []
for _ in range(2000):
    xt = rng.choice(x, size=len(x), replace=True)
    yc = rng.choice(y, size=len(y), replace=True)
    boot_diffs.append(xt.mean() - yc.mean())

bootstrap_ci = np.percentile(boot_diffs, [2.5, 97.5])
bootstrap_width = bootstrap_ci[1] - bootstrap_ci[0]

# 4. Matching (задача 11)
features = [
    "age", "tenure_days", "kyc_level",
    "members", "spend_prev", "sessions_prev"
]

match_data = mine[
    ["client_id", "grp", "spend"] + features
].copy()
match_data["t"] = (match_data["grp"] == "test").astype(int)
match_data = match_data.drop(columns="grp")

roles = {
    "client_id": InfoRole(str),
    "t": TreatmentRole(),
    "spend": TargetRole(),
    **{col: FeatureRole() for col in features}
}

ds_match = Dataset(
    roles=roles,
    data=match_data,
    default_role=FeatureRole()
)

matching_result = Matching(n_neighbors=5).execute(ds_match).resume

# Получаем обычный pandas DataFrame из внутреннего объекта HypEx
matching_df = matching_result.backend.data

print("Колонки результата:", matching_df.columns.tolist())
print(matching_df)

ate = matching_df.loc[matching_df.index == "ATE"].iloc[0]

matching_ci = (
    float(ate["CI Lower"]),
    float(ate["CI Upper"])
)
matching_width = matching_ci[1] - matching_ci[0]

# 5. Экономический результат (задача 12)
tickets_effect = diff_ci(
    test["tickets"], control["tickets"]
)["diff"]

churn_effect = diff_ci(
    test["churned"], control["churned"]
)["diff"]

commission_income = cuped_effect["diff"] * COMMISSION
tickets_cost = tickets_effect * TICKET_COST
churn_cost = churn_effect * CHURN_COST

total = commission_income - tickets_cost - churn_cost

# 6. Корреляция purchases — проверка утечки данных
purchases_r2 = mine["spend"].corr(mine["purchases"]) ** 2 * 100

# Вывод результатов
print("СРАВНЕНИЕ 95%-Х ДОВЕРИТЕЛЬНЫХ ИНТЕРВАЛОВ")
print("-" * 65)

intervals = {
    "Задача 3 — обычная оценка": (raw_ci, raw_width),
    "Задача 7 — CUPED": (cuped_ci, cuped_width),
    "Задача 9 — Bootstrap": (bootstrap_ci, bootstrap_width),
    "Задача 11 — Matching": (matching_ci, matching_width)
}

for name, (ci, width) in intervals.items():
    print(
        f"{name}: [{ci[0]:.2f}; {ci[1]:.2f}] ₽"
        f" | ширина: {width:.2f} ₽"
    )

print("\nЭКОНОМИКА АКЦИИ")
print("-" * 65)
print(f"Комиссия: {commission_income:.2f} ₽")
print(f"Обращения в поддержку: {-tickets_cost:.2f} ₽")
print(f"Отток: {-churn_cost:.2f} ₽")
print(f"Итог: {total:.2f} ₽ на клиента")

print("\nПРОВЕРКА КОВАРИАТЫ")
print("-" * 65)
print(f"R² для purchases: {purchases_r2:.1f}%")

question_3 = {
    "interval_widths": {
        name: round(width, 2)
        for name, (_, width) in intervals.items()
    },
    "profit_per_client": round(float(total), 2)
}

question_3

Колонки результата: ['Effect Size', 'Standard Error', 'P-value', 'CI Lower', 'CI Upper', 'outcome']
     Effect Size  Standard Error  P-value  CI Lower  CI Upper outcome
ATT        32.42           72.01     0.65   -108.73    173.56   spend
ATC        41.18           72.02     0.57    -99.98    182.33   spend
ATE        36.77           69.62     0.60    -99.69    173.23   spend
СРАВНЕНИЕ 95%-Х ДОВЕРИТЕЛЬНЫХ ИНТЕРВАЛОВ
-----------------------------------------------------------------
Задача 3 — обычная оценка: [-132.12; 196.74] ₽ | ширина: 328.85 ₽
Задача 7 — CUPED: [-87.94; 177.91] ₽ | ширина: 265.84 ₽
Задача 9 — Bootstrap: [-136.22; 206.24] ₽ | ширина: 342.45 ₽
Задача 11 — Matching: [-99.69; 173.23] ₽ | ширина: 272.92 ₽

ЭКОНОМИКА АКЦИИ
-----------------------------------------------------------------
Комиссия: 0.67 ₽
Обращения в поддержку: 1.06 ₽
Отток: -0.50 ₽
Итог: 1.22 ₽ на клиента

ПРОВЕРКА КОВАРИАТЫ
-----------------------------------------------------------------
R² для purchase

{'interval_widths': {'Задача 3 — обычная оценка': 328.85,
  'Задача 7 — CUPED': 265.84,
  'Задача 9 — Bootstrap': 342.45,
  'Задача 11 — Matching': 272.92},
 'profit_per_client': 1.22}

#### ✍️ Ответ на вопрос 3

Больше всего я верю оценке из задачи 7: 44,98 ₽ с интервалом от −87,94 до 177,91 ₽, ширина 265,85 ₽ против 328,85 ₽ у обычной оценки и 342,46 ₽ у бутстрапа. Бутстреп шире, потому что по тяжённому распределению оборота нормальное приближение немного оптимистично, и расхождение интервалов задаёт честный масштаб неопределённости; мэтчинг (ATE 36,77 ₽, ширина 272,92 ₽) близка к CUPED, но в рандомизированном эксперименте он ничего не добавляет и лишь подбирает соседей по признакам вместо прямой ковариаты. purchases как ковариату брать нельзя, хотя ρ² = 70,8% и прирост самый большой: эта колонка измерена за время эксперимента и сама могла измениться от акции, то есть часть эффекта переехала бы в поправку и оценка исказилась. В деньгах по допущениям задачи 12 вышло +1,22 ₽ на клиента за 4 недели: 0,67 ₽ комиссии, плюс 1,06 ₽ экономии на обращениях, минус 0,50 ₽ потерь от оттока, то есть почти вся прибыль это экономия на поддержке, а не рост оборота. Это выглядит окупаемым, но в расчёте нет самого дорогого — выплата бонуса 300 ₽ за первую покупку в доставке (при комиссии 1,5% бонус окупается примерно при обороте 20 000 ₽), и если даже 5% клиентов его заберут, это минус около 15 ₽ на клиента против наших 1,22 ₽; плюс сам интервал в пересчёте на деньги лежит от −0,77 до +3,22 ₽, то есть знак тоже не доказан.

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту

**Что проверяли:** Четырёхнедельный A/B-тест акции «Бонус 300 ₽ за первую покупку в доставке» на 6 000 аккаунтов подписки, поровну тест и контроль. Смотрели оборот в партнёрских сервисах как основную метрику и отток с обращениями в поддержку как стражей; перед анализом проверили, что группы разделились честно и были похожи по обороту до старта.

**Что получилось:** Оборот в тесте выше на 32,31 ₽, после CUPED на 44,98 ₽ на клиента за 4 недели, 95% интервал от −87,94 до 177,91 ₽, p = 0,507. Ноль внутри интервала: мы не можем ни сказать, что акция сработала, ни доказать, что она не работает. Прирост в 96 ₽, ради которого всё затевали интервал не исключает. В деньгах по нашим допущениям +1,22 ₽ на клиента, но это точечная оценка, и в интервале она от −0,77 до +3,22 ₽, то есть на всю базу от минус 4,6 до плюс 19,3 тыс. ₽ за месяц.

**Какие остались вопросы:** Группы сошлись (3 017 к 2 983, p = 0,661), до старта по обороту не различались (p = 0,797), стражи не ухудшились: отток +0,07 п.п. (p = 0,937), обращений на 0,007 на клиента меньше (p = 0,274), после поправки Холма не значимо ничего. Главная проблема - чувствительность: эксперимент на 3 000 клиентов на группу замечает 187 ₽ с CUPED, а мы ждали 96 ₽, мощность на такой эффект около 30%. И ещё один большой вопрос - экономика без бонуса: расчёт задачи 12 не учитывает выплату 300 ₽, из-за чего текущая оценка держится в основном на экономии 1,06 ₽ на поддержке.

**Что рекомендуем:** Не масштабировать. По решающему правилу нам нужна нижняя граница интервала выше нуля и по эффекту, и в деньгах, а сейчас обе ниже нуля, поэтому запуск был бы ставкой на удачу. Повторять тот же дизайн с той же выборкой тоже бессмысленно - мощности хватило бы примерно на треть: под 96 ₽ нужно около 22 600 аккаунтов на группу с CUPED либо более крупный эффект. Сначала считаем акцию целиком с бонусом и расходами на доставку (сейчас при 1,5% комиссии бонус окупается только при обороте 20 000 ₽), и если после этого экономика сходится с запасом - запускаем тест на ~23 тысячи аккаунтов на группу, стражи те же.

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [22]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

задача  1: 0.661
    ✅ формат в порядке
задача  2: 0.797
    ✅ формат в порядке
задача  3: [32.31, -132.12, 196.74]
    ✅ формат в порядке
задача  4: [0.7001, 0.5824]
    ✅ формат в порядке
задача  5: [0.07, 0.937, -0.007, 0.274]
    ✅ формат в порядке
задача  6: []
    ✅ формат в порядке
задача  7: [0.598, 34.7, 44.98, -87.94, 177.91]
    ✅ формат в порядке
задача  8: 44.98
    ✅ формат в порядке
задача  9: [-136.22, 206.24]
    ✅ формат в порядке
задача 10: ['spend_prev+spend_prev2', 41.5]
    ✅ формат в порядке
задача 11: [36.77, -99.69, 173.23]
    ✅ формат в порядке
задача 12: 1.22
    ✅ формат в порядке

Не забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.

--- служебная строка, не удаляйте ---
HW3_ANSWERS={"variant": 53, "answers": {"1": 0.661, "2": 0.797, "3": [32.31, -132.12, 196.74], "4": [0.7001, 0.5824], "5": [0.07, 0.937, -0.007, 0.274], "6": [], "7": [0.598, 34.7, 44.98, -87.94, 177.91], "8": 44.98, "9": [-136.22, 206.24], "10": ["spend_prev+spend_p

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [23]:
# Исследование А: ваш код. Всё — от mine.

#### ✍️ Исследование А

_Замените эту строку своим текстом, заголовок выше не трогайте. Схема: вопрос → как проверяли → что получилось → что это значит и чего расчёт не доказывает._

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [24]:
# Исследование Б: ваш код. Всё — от mine.

#### ✍️ Исследование Б

_Замените эту строку своим текстом, заголовок выше не трогайте. Схема: вопрос → как проверяли → что получилось → что это значит и чего расчёт не доказывает._